This is to plot differences between different LLM.

This is after finetunePro_starmap_local.ipynb
all the metrics are calculated in SDMBench project

Note: 
* in BZ5, zeroshot for gpt4omini use all cells. other LLM use 70% of cells.
* all fine tuning results are stage one results



In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42
import seaborn as sns
import numpy as np
from matplotlib.patches import Patch

# STARmap

In [ ]:
# zeroshot
original_results_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv', index_col=0)
original_results_df = original_results_df[original_results_df['data_type'] == 'starmap']
original_results_df = original_results_df[original_results_df['model_name'] == 'gpt4o_mini']
original_results_df['num_niches'] = 4
original_results_df['experiment_types'] = 'zeroshot'

# finetune
finetune_results_df = pd.read_csv('examples/results/finetunePro_all_metrics_all_replicates.csv', index_col=0)
finetune_results_df = finetune_results_df[finetune_results_df['data_type'] == 'starmap']
finetune_results_df = finetune_results_df[finetune_results_df['stage'] == 'one']
finetune_results_df['num_niches'] = 4
finetune_results_df['experiment_types'] = 'finetune_BZ5'
finetune_results_df['model_name'] = 'gpt4o_mini'

# local LLM
local_LLM_all_metrics = pd.read_csv("examples/results/localllm_starmap_all_metrics.csv", index_col=0)
# rename data_name from BZ5_test to BZ5, use string split   
local_LLM_all_metrics['data_name'] = local_LLM_all_metrics['data_name'].str.split('_').str[0]


In [ ]:
column_to_compare = ['data_name', 'model_name', 'experiment_types', 'replicate',
       'ARI', 'NMI', 'HOM', 'COM', 'CHAOS', 'PAS', 'ASW']
all_results_df = pd.concat([original_results_df[column_to_compare].copy(), 
                            finetune_results_df[column_to_compare].copy(), 
                            local_LLM_all_metrics[column_to_compare].copy()])

In [ ]:
possible_metrics = ['ARI', 'NMI', 'COM']

In [ ]:
all_results_df.head()

In [ ]:
all_results_df.model_name.unique()

In [ ]:

def plot_metric_comparison(all_results_df, metric, experiment_type, 
                          data_names = ['BZ5', 'BZ9', 'BZ14'],
                          model_names = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini'], 
                          figsize=(12, 6), palette='Set2', show_values=False, ylim=None):
    """
    Plot barplot with error bars comparing different models for each dataset.
    
    Parameters:
    -----------
    all_results_df : pd.DataFrame
        DataFrame containing columns: data_name, model_name, experiment_types, and metric columns
    metric : str
        The metric to plot (e.g., 'ARI', 'NMI', 'HOM', etc.)
    experiment_type : str
        The experiment type to filter for (e.g., 'zeroshot', 'finetune_BZ5')
    figsize : tuple, optional
        Figure size (width, height)
    palette : str, optional
        Color palette for different models
    show_values : bool, optional
        Whether to show values on top of bars
    ylim : tuple, optional
        Y-axis limits (min, max)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axis objects
    """
    # Filter data for the specified experiment type
    filtered_df = all_results_df[all_results_df['experiment_types'] == experiment_type].copy()
    
    if filtered_df.empty:
        print(f"No data found for experiment_type: {experiment_type}")
        return None, None
    
    # Calculate mean and std for each data_name and model_name combination
    summary_stats = filtered_df.groupby(['data_name', 'model_name'])[metric].agg(['mean', 'std', 'count']).reset_index()
    
    # Get unique data names and model names
    
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Set up bar positions
    x = np.arange(len(data_names))
    width = 0.8 / len(model_names)  # Width of bars
    
    # Create color palette
    colors = sns.color_palette(palette, len(model_names))
    
    # Plot bars for each model
    for i, model in enumerate(model_names):
        model_data = summary_stats[summary_stats['model_name'] == model]
        
        means = []
        stds = []
        for data_name in data_names:
            data_point = model_data[model_data['data_name'] == data_name]
            if not data_point.empty:
                means.append(data_point['mean'].values[0])
                stds.append(data_point['std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        
        # Plot bars with error bars
        positions = x + (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(positions, means, width, yerr=stds, 
                     label=model, color=colors[i], alpha=0.8,
                     capsize=5, error_kw={'linewidth': 1.5})
        
        # Optionally show values on top of bars
        if show_values:
            for j, (bar, mean, std) in enumerate(zip(bars, means, stds)):
                height = bar.get_height()
                if mean > 0:  # Only show if there's data
                    ax.text(bar.get_x() + bar.get_width()/2., height + std,
                           f'{mean:.3f}',
                           ha='center', va='bottom', fontsize=8, rotation=0)
    
    # Customize plot
    ax.set_xlabel('Dataset', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison Across Models\n({experiment_type})', 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(data_names, rotation=45, ha='right')
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Set y-axis limits if provided
    if ylim is not None:
        ax.set_ylim(ylim)
    
    # Add a horizontal line at y=0 for reference
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    
    return fig, ax


In [ ]:
# Example 1: Plot ARI for zeroshot experiment
metric_type = "NMI"
fig, ax = plot_metric_comparison(all_results_df, 
                                 metric=metric_type, 
                                 experiment_type='zeroshot',
                                 model_names = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini'],
                                 figsize=(10, 6),
                                 ylim=(0, 1))
# save
plt.savefig(f"figures/{metric_type}_local_LLM.pdf")
plt.show()


In [ ]:
# Example 1: Plot ARI for zeroshot experiment
fig, ax = plot_metric_comparison(all_results_df, 
                                 metric='NMI', 
                                 experiment_type='finetune_BZ5',
                                 model_names = ["Llama8", "Llama8_nothink", "Qwen30", "Qwen30_nothink", "Llama70", "Llama70_nothink", "gpt4o_mini"],  #  "Llama70_6e", "Llama8_nothink", "Qwen30_nothink", "Llama70_nothink", 'gpt4o_mini'
                                 figsize=(10, 6),
                                 ylim=(0, 1))
plt.show()

In [ ]:
# Example 3: Plot all metrics for a specific experiment type
experiment_type = 'finetune_BZ5'
metrics = ['ARI', 'NMI',  'COM']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Filter data for the specified experiment type
    filtered_df = all_results_df[all_results_df['experiment_types'] == experiment_type].copy()
    
    # Calculate mean and std for each data_name and model_name combination
    summary_stats = filtered_df.groupby(['data_name', 'model_name'])[metric].agg(['mean', 'std']).reset_index()
    
    # Get unique data names and model names
    data_names = ['BZ5', 'BZ9', 'BZ14']
    model_names = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    # Set up bar positions
    x = np.arange(len(data_names))
    width = 0.8 / len(model_names)
    
    # Create color palette
    colors = sns.color_palette('Set2', len(model_names))
    
    # Plot bars for each model
    for j, model in enumerate(model_names):
        model_data = summary_stats[summary_stats['model_name'] == model]
        
        means = []
        stds = []
        for data_name in data_names:
            data_point = model_data[model_data['data_name'] == data_name]
            if not data_point.empty:
                means.append(data_point['mean'].values[0])
                stds.append(data_point['std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        
        # Plot bars with error bars
        positions = x + (j - len(model_names)/2 + 0.5) * width
        ax.bar(positions, means, width, yerr=stds, 
               label=model, color=colors[j], alpha=0.8,
               capsize=3, error_kw={'linewidth': 1})
    
    # Customize subplot
    ax.set_xlabel('Dataset', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(data_names, rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if i == 0:  # Only show legend on first subplot
        ax.legend(title='Model', fontsize=8, title_fontsize=9)

# Remove extra subplots
for i in range(len(metrics), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle(f'All Metrics Comparison - {experiment_type}', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
def plot_boxplot_comparison_by_dataset(all_results_df, metric, 
                                       baseline_models=None, compare_models=None,
                                       baseline_exp='zeroshot', compare_exp='finetune_BZ5',
                                       figsize=(15, 5), palette=None, ylim=None, show_points=True):
    """
    Plot boxplot comparing two experiment types (baseline vs comparison) across different models with pair-to-pair comparison.
    Creates separate subplots for each dataset (data_name).
    
    Parameters:
    -----------
    all_results_df : pd.DataFrame
        DataFrame containing columns: data_name, model_name, experiment_types, and metric columns
    metric : str
        The metric to plot (e.g., 'ARI', 'NMI', 'HOM', etc.)
    baseline_models : list, optional
        List of model names to use for baseline experiment.
        If None, uses default: ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    compare_models : list, optional
        List of model names to use for comparison experiment. Should have same length as baseline_models.
        If None, uses the same models as baseline_models.
    baseline_exp : str, optional
        The baseline experiment type name in the dataframe (default: 'zeroshot')
    compare_exp : str, optional
        The comparison experiment type name in the dataframe (default: 'finetune_BZ5')
    figsize : tuple, optional
        Figure size (width, height)
    palette : dict or str, optional
        Color palette for experiment types. If dict, keys should be experiment type names (values of baseline_exp and compare_exp).
    ylim : tuple, optional
        Y-axis limits (min, max)
    show_points : bool, optional
        Whether to show individual data points on top of boxes
    
    Returns:
    --------
    fig, axes : matplotlib figure and axis objects
    """
    # Set default model lists if not provided
    if baseline_models is None:
        baseline_models = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    if compare_models is None:
        compare_models = baseline_models.copy()
    
    # Validate that both lists have the same length
    if len(baseline_models) != len(compare_models):
        raise ValueError(f"baseline_models and compare_models must have the same length. "
                        f"Got {len(baseline_models)} and {len(compare_models)} respectively.")
    
    # Filter data for the two experiment types
    filtered_df = all_results_df[
        all_results_df['experiment_types'].isin([baseline_exp, compare_exp])
    ].copy()
    
    if filtered_df.empty:
        print(f"No data found for experiment types: {baseline_exp} or {compare_exp}")
        return None, None
    
    # Get unique datasets
    # You can update this list if your dataset names change
    data_names = ['BZ5', 'BZ9', 'BZ14']
    n_datasets = len(data_names)
    
    # Create subplots
    fig, axes = plt.subplots(1, n_datasets, figsize=figsize, sharey=True)
    if n_datasets == 1:
        axes = [axes]
    
    # Define colors
    if palette is None:
        palette = {
            baseline_exp: '#8dd3c7',
            compare_exp: '#fb8072'
        }
    
    # Plot each dataset
    for idx, data_name in enumerate(data_names):
        ax = axes[idx]
        
        # Filter for this dataset
        dataset_df = filtered_df[filtered_df['data_name'] == data_name].copy()
        
        # Check if data exists for this dataset
        if dataset_df.empty:
            continue

        # Create a combined column for x-axis grouping
        dataset_df['model_exp'] = dataset_df['model_name'] + '\n' + dataset_df['experiment_types']
        
        # Define the order for x-axis (alternating baseline and comparison for each model pair)
        x_order = []
        for b_model, c_model in zip(baseline_models, compare_models):
            x_order.append(f"{b_model}\n{baseline_exp}")
            x_order.append(f"{c_model}\n{compare_exp}")
        
        # Filter to only include models that exist in the data
        x_order = [x for x in x_order if x in dataset_df['model_exp'].values]
        
        # Create color list for each box based on experiment type
        colors = []
        for x in x_order:
            if baseline_exp in x:
                colors.append(palette.get(baseline_exp, '#8dd3c7'))
            else:
                colors.append(palette.get(compare_exp, '#fb8072'))
        
        if not x_order:
            continue
            
        # Create boxplot
        box_data = [dataset_df[dataset_df['model_exp'] == x][metric].values for x in x_order]
        
        bp = ax.boxplot(box_data, 
                        labels=x_order,
                        patch_artist=True,
                        widths=0.6,
                        showmeans=True,
                        meanprops=dict(marker='D', markerfacecolor='red', markersize=6, 
                                      markeredgecolor='darkred', linewidth=1.5),
                        medianprops=dict(color='black', linewidth=2),
                        boxprops=dict(linewidth=1.5),
                        whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5))
        
        # Color the boxes
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Optionally add individual points
        if show_points:
            for i, x in enumerate(x_order):
                data = dataset_df[dataset_df['model_exp'] == x][metric].values
                # Add jitter to x-coordinates
                x_coords = np.random.normal(i + 1, 0.04, size=len(data))
                ax.scatter(x_coords, data, alpha=0.4, s=30, color='black', zorder=3)
        
        # Customize x-axis labels to show model names more clearly
        x_labels = []
        for label in x_order:
            model = label.split('\n')[0]
            exp_type_val = label.split('\n')[1]
            
            # Create short labels
            if exp_type_val == 'zeroshot':
                short_exp = 'ZS'
            elif 'finetune' in exp_type_val:
                short_exp = 'FT'
            elif exp_type_val == 'ttrlgc':
                short_exp = 'TTRL-GC'
            else:
                # Fallback: take first 4 chars uppercase or just use the name
                short_exp = exp_type_val[:4].upper() if len(exp_type_val) > 4 else exp_type_val
            
            x_labels.append(f"{model}\n({short_exp})")
        
        ax.set_xticklabels(x_labels, fontsize=9)
        
        # Add vertical lines to separate models
        for i in range(2, len(x_order), 2):
            ax.axvline(x=i + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
        
        # Customize subplot
        ax.set_title(f'{data_name}', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Set y-axis limits if provided
        if ylim is not None:
            ax.set_ylim(ylim)
        
        # Only add y-label to leftmost subplot
        if idx == 0:
            ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    
    # Create custom legend on the rightmost subplot
    legend_elements = [
        Patch(facecolor=palette.get(baseline_exp, '#8dd3c7'), alpha=0.7, label=baseline_exp),
        Patch(facecolor=palette.get(compare_exp, '#fb8072'), alpha=0.7, label=compare_exp),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='red', 
                   markersize=8, markeredgecolor='darkred', label='Mean')
    ]
    axes[-1].legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    # Add overall title
    fig.suptitle(f'{metric} Comparison: {baseline_exp} vs {compare_exp} by Dataset', 
                fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    return fig, axes

In [ ]:
fig, ax = plot_boxplot_comparison_by_dataset(all_results_df, 
                                  metric='NMI',
                                  baseline_models=['Qwen2.5-7'],
                                  compare_exp='ttrlgc',
                                  figsize=(12, 6),
                                  ylim=(0, 1),
                                  show_points=False)
plt.show()

In [ ]:
def plot_boxplot_comparison(all_results_df, metric, 
                            baseline_models=None, compare_models=None,
                            baseline_exp='zeroshot', compare_exp='finetune_BZ5',
                            figsize=(10, 6), palette=None, ylim=None, show_points=True):
    """
    Plot boxplot comparing two experiment types (baseline vs comparison) across different models.
    Creates paired boxes for each model entry.
    
    Parameters:
    -----------
    all_results_df : pd.DataFrame
        DataFrame containing columns: data_name, model_name, experiment_types, and metric columns
    metric : str
        The metric to plot (e.g., 'ARI', 'NMI', 'HOM', etc.)
    baseline_models : list, optional
        List of model names to use for baseline experiment.
        If None, uses default: ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    compare_models : list, optional
        List of model names to use for comparison experiment. Should have same length as baseline_models.
        If None, uses the same models as baseline_models.
    baseline_exp : str, optional
        The baseline experiment type name in the dataframe (default: 'zeroshot')
    compare_exp : str, optional
        The comparison experiment type name in the dataframe (default: 'finetune_BZ5')
    figsize : tuple, optional
        Figure size (width, height)
    palette : dict or str, optional
        Color palette for experiment types. If dict, keys should be experiment type names.
    ylim : tuple, optional
        Y-axis limits (min, max)
    show_points : bool, optional
        Whether to show individual data points on top of boxes
    
    Returns:
    --------
    fig, ax : matplotlib figure and axis objects
    """
    # Set default model lists if not provided
    if baseline_models is None:
        baseline_models = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    if compare_models is None:
        compare_models = baseline_models.copy()
    
    # Validate that both lists have the same length
    if len(baseline_models) != len(compare_models):
        raise ValueError(f"baseline_models and compare_models must have the same length. "
                        f"Got {len(baseline_models)} and {len(compare_models)} respectively.")
    
    # Filter data for experiment types
    filtered_df = all_results_df[
        all_results_df['experiment_types'].isin([baseline_exp, compare_exp])
    ].copy()
    
    if filtered_df.empty:
        print(f"No data found for experiment types: {baseline_exp} or {compare_exp}")
        return None, None
    
    # Create a combined column for x-axis grouping
    filtered_df['model_exp'] = filtered_df['model_name'] + '\n' + filtered_df['experiment_types']
    
    # Define the order for x-axis (alternating baseline and comparison for each model pair)
    x_order = []
    for b_model, c_model in zip(baseline_models, compare_models):
        x_order.append(f"{b_model}\n{baseline_exp}")
        x_order.append(f"{c_model}\n{compare_exp}")
    
    # Filter to only include models that exist in the data
    # Note: If a model/exp combo is missing entirely, it will be skipped, potentially breaking the visual pairing if only one of the pair exists.
    # However, keeping only existing values prevents crashes.
    x_order = [x for x in x_order if x in filtered_df['model_exp'].values]
    
    if not x_order:
        print("No matching data found for the specified models and experiment types.")
        return None, None

    # Set up the plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Define colors
    if palette is None:
        palette = {
            baseline_exp: '#8dd3c7',
            compare_exp: '#fb8072'
        }
    
    # Create color list for each box based on experiment type
    colors = []
    for x in x_order:
        if baseline_exp in x:
            colors.append(palette.get(baseline_exp, '#8dd3c7'))
        else:
            colors.append(palette.get(compare_exp, '#fb8072'))
    
    # Create boxplot
    box_data = [filtered_df[filtered_df['model_exp'] == x][metric].values for x in x_order]
    
    bp = ax.boxplot(box_data, 
                    labels=x_order,
                    patch_artist=True,
                    widths=0.6,
                    showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=6, 
                                  markeredgecolor='darkred', linewidth=1.5),
                    medianprops=dict(color='black', linewidth=2),
                    boxprops=dict(linewidth=1.5),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Optionally add individual points
    if show_points:
        for i, x in enumerate(x_order):
            data = filtered_df[filtered_df['model_exp'] == x][metric].values
            # Add jitter to x-coordinates
            x_coords = np.random.normal(i + 1, 0.04, size=len(data))
            ax.scatter(x_coords, data, alpha=0.4, s=30, color='black', zorder=3)
    
    # Customize x-axis labels to show model names more clearly
    x_labels = []
    for i, label in enumerate(x_order):
        model = label.split('\n')[0]
        exp_type_val = label.split('\n')[1]
        
        # Helper to shorten common experiment names for labels
        if exp_type_val == 'zeroshot':
            short_exp = 'ZS'
        elif 'finetune' in exp_type_val:
            short_exp = 'FT'
        elif exp_type_val == 'ttrlgc':
            short_exp = 'TTRL'
        else:
             # Fallback: take first 4 chars uppercase
            short_exp = exp_type_val[:4].upper() if len(exp_type_val) > 4 else exp_type_val
            
        x_labels.append(f"{model}\n({short_exp})")
    
    ax.set_xticklabels(x_labels, fontsize=10)
    
    # Add vertical lines to separate models (every pair)
    # We assume pairs, but if data is missing, this might look odd. 
    # Drawing lines between every 2 items is a safe default for visual grouping.
    for i in range(2, len(x_order), 2):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
    
    # Customize plot
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison: {baseline_exp} vs {compare_exp}', 
                fontsize=14, fontweight='bold', pad=20)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Set y-axis limits if provided
    if ylim is not None:
        ax.set_ylim(ylim)
    
    # Create custom legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=palette.get(baseline_exp, '#8dd3c7'), alpha=0.7, label=baseline_exp),
        Patch(facecolor=palette.get(compare_exp, '#fb8072'), alpha=0.7, label=compare_exp),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='red', 
                   markersize=8, markeredgecolor='darkred', label='Mean')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    
    return fig, ax

In [ ]:
fig, ax = plot_boxplot_comparison(all_results_df, 
                                  metric='NMI',
                                  figsize=(12, 6),
                                  ylim=(0, 1),
                                  show_points=False)
plt.show()

In [ ]:
# Example 4: Boxplot comparison separated by dataset
fig, axes = plot_boxplot_comparison_by_dataset(all_results_df, 
                                               metric='NMI',
                                               figsize=(18, 6),
                                               ylim=(0.4, 0.9),
                                               show_points=False)
plt.show()


In [ ]:
# Example 4b: Boxplot comparison with custom model pairs
# Compare zeroshot of Llama8 and Llama70 to finetune_BZ5 of Llama8 and Llama70_nothink
fig, axes = plot_boxplot_comparison_by_dataset(all_results_df, 
                                               metric='NMI',
                                               zeroshot_models=[ 'Llama70'],
                                               finetune_models=['Llama70_ZSprompt'],
                                               figsize=(18, 6),
                                               ylim=(0.4, 0.9),
                                               show_points=False)
plt.show()


In [ ]:
# Example 6: Create boxplots for all metrics in a grid
metrics = ['ARI', 'NMI',  'COM' ]

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Filter data for zeroshot and finetune_BZ5
    filtered_df = all_results_df[
        all_results_df['experiment_types'].isin(['zeroshot', 'finetune_BZ5'])
    ].copy()
    
    # Define custom order for models
    model_order = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    # Create a combined column for x-axis grouping
    filtered_df['model_exp'] = filtered_df['model_name'] + '\n' + filtered_df['experiment_types']
    
    # Define the order for x-axis
    x_order = []
    for model in model_order:
        x_order.append(f"{model}\nzeroshot")
        x_order.append(f"{model}\nfinetune_BZ5")
    
    # Filter to only include models that exist in the data
    x_order = [x for x in x_order if x in filtered_df['model_exp'].values]
    
    # Define colors
    palette = {
        'zeroshot': '#8dd3c7',
        'finetune_BZ5': '#fb8072'
    }
    
    colors = []
    for x in x_order:
        if 'zeroshot' in x:
            colors.append(palette['zeroshot'])
        else:
            colors.append(palette['finetune_BZ5'])
    
    # Create boxplot
    box_data = [filtered_df[filtered_df['model_exp'] == x][metric].values for x in x_order]
    
    bp = ax.boxplot(box_data, 
                    labels=x_order,
                    patch_artist=True,
                    widths=0.6,
                    showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=4, 
                                  markeredgecolor='darkred', linewidth=1),
                    medianprops=dict(color='black', linewidth=1.5),
                    boxprops=dict(linewidth=1),
                    whiskerprops=dict(linewidth=1),
                    capprops=dict(linewidth=1))
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Customize x-axis labels
    x_labels = []
    for label in x_order:
        model = label.split('\n')[0]
        exp_type = 'ZS' if 'zeroshot' in label else 'FT'
        x_labels.append(f"{model}\n({exp_type})")
    
    ax.set_xticklabels(x_labels, fontsize=8, rotation=0)
    
    # Add vertical lines to separate models
    for j in range(2, len(x_order), 2):
        ax.axvline(x=j + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
    
    # Customize subplot
    ax.set_ylabel(metric, fontsize=10, fontweight='bold')
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add legend to first subplot
    if i == 0:
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=palette['zeroshot'], alpha=0.7, label='Zero-shot'),
            Patch(facecolor=palette['finetune_BZ5'], alpha=0.7, label='Fine-tuned'),
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    ax.set_ylim(0,0.9)

# Remove extra subplots
for i in range(len(metrics), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('All Metrics: Zero-shot vs Fine-tuned Comparison', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# Example 7: Create boxplots for all metrics, separated by dataset
metrics = ['ARI', 'NMI',  'COM']

fig, axes = plt.subplots(len(metrics), 1, figsize=(18, 6 * len(metrics)))

for i, metric in enumerate(metrics):
    # Filter data for zeroshot and finetune_BZ5
    filtered_df = all_results_df[
        all_results_df['experiment_types'].isin(['zeroshot', 'finetune_BZ5'])
    ].copy()
    
    # Get unique datasets
    data_names = ['BZ5', 'BZ9', 'BZ14']
    n_datasets = len(data_names)
    
    # Define custom order for models
    model_order = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    # Define colors
    palette = {
        'zeroshot': '#8dd3c7',
        'finetune_BZ5': '#fb8072'
    }
    
    # Create a single row of subplots for this metric
    if i == 0:
        # For first metric, create the structure
        metric_axes = [plt.subplot(len(metrics), n_datasets, i * n_datasets + j + 1) 
                      for j in range(n_datasets)]
    else:
        metric_axes = [plt.subplot(len(metrics), n_datasets, i * n_datasets + j + 1) 
                      for j in range(n_datasets)]
    
    # Plot each dataset
    for idx, data_name in enumerate(data_names):
        ax = metric_axes[idx]
        
        # Filter for this dataset
        dataset_df = filtered_df[filtered_df['data_name'] == data_name].copy()
        
        # Create a combined column for x-axis grouping
        dataset_df['model_exp'] = dataset_df['model_name'] + '\n' + dataset_df['experiment_types']
        
        # Define the order for x-axis
        x_order = []
        for model in model_order:
            x_order.append(f"{model}\nzeroshot")
            x_order.append(f"{model}\nfinetune_BZ5")
        
        # Filter to only include models that exist in the data
        x_order = [x for x in x_order if x in dataset_df['model_exp'].values]
        
        # Create color list
        colors = []
        for x in x_order:
            if 'zeroshot' in x:
                colors.append(palette['zeroshot'])
            else:
                colors.append(palette['finetune_BZ5'])
        
        # Create boxplot
        box_data = [dataset_df[dataset_df['model_exp'] == x][metric].values for x in x_order]
        
        bp = ax.boxplot(box_data, 
                        labels=x_order,
                        patch_artist=True,
                        widths=0.6,
                        showmeans=True,
                        meanprops=dict(marker='D', markerfacecolor='red', markersize=4, 
                                      markeredgecolor='darkred', linewidth=1),
                        medianprops=dict(color='black', linewidth=1.5),
                        boxprops=dict(linewidth=1),
                        whiskerprops=dict(linewidth=1),
                        capprops=dict(linewidth=1))
        
        # Color the boxes
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Customize x-axis labels
        x_labels = []
        for label in x_order:
            model = label.split('\n')[0]
            exp_type = 'ZS' if 'zeroshot' in label else 'FT'
            x_labels.append(f"{model}\n({exp_type})")
        
        ax.set_xticklabels(x_labels, fontsize=8)
        
        # Add vertical lines to separate models
        for j in range(2, len(x_order), 2):
            ax.axvline(x=j + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
        
        # Customize subplot
        if i == 0:  # Only show dataset name on top row
            ax.set_title(f'{data_name}', fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Only add y-label to leftmost subplot
        if idx == 0:
            ax.set_ylabel(metric, fontsize=11, fontweight='bold')
        
        # Add legend to top-right subplot
        if i == 0 and idx == n_datasets - 1:
            from matplotlib.patches import Patch
            legend_elements = [
                Patch(facecolor=palette['zeroshot'], alpha=0.7, label='Zero-shot'),
                Patch(facecolor=palette['finetune_BZ5'], alpha=0.7, label='Fine-tuned'),
            ]
            ax.legend(handles=legend_elements, fontsize=8)
        
        ax.set_ylim(0,0.9)
    

plt.suptitle('Metrics Comparison: Zero-shot vs Fine-tuned by Dataset', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


# LIBD

In [ ]:
# zeroshot
original_results_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv', index_col=0)
original_results_df = original_results_df[original_results_df['data_type'] == 'visium']
original_results_df = original_results_df[original_results_df['model_name'] == 'gpt4o_mini']
original_results_df['experiment_types'] = 'zeroshot'

# finetune
finetune_results_df = pd.read_csv('examples/results/finetunePro_all_metrics_all_replicates.csv', index_col=0)
finetune_results_df = finetune_results_df[finetune_results_df['data_type'] == 'visium']
finetune_results_df = finetune_results_df[finetune_results_df['stage'] == 'one']
finetune_results_df = finetune_results_df[finetune_results_df['finetune_name'] == '151673']
finetune_results_df['experiment_types'] = 'finetune_151673'
finetune_results_df['model_name'] = 'gpt4o_mini'

# local LLM
local_LLM_all_metrics = pd.read_csv("examples/results/localllm_libd_all_metrics.csv", index_col=0)
local_LLM_all_metrics['data_name'] = local_LLM_all_metrics['data_name'].str.split('_').str[0]
local_LLM_all_metrics['data_type'] = 'visium'


In [ ]:
column_to_compare = ['data_name', 'model_name', 'experiment_types', 'replicate',
       'ARI', 'NMI', 'HOM', 'COM', 'CHAOS', 'PAS', 'ASW']
all_results_df = pd.concat([original_results_df[column_to_compare].copy(), 
                            finetune_results_df[column_to_compare].copy(), 
                            local_LLM_all_metrics[column_to_compare].copy()])

In [ ]:
all_results_df

In [ ]:
fig, ax = plot_metric_comparison(all_results_df, 
                                 metric='COM', 
                                 experiment_type='zeroshot',
                                 data_names = ['151673', '151674', '151675', '151676'],
                                 model_names = ["Llama8", "Qwen30", "Llama70", 'gpt4o_mini'],
                                 figsize=(10, 6),
                                 ylim=(0, 1))
plt.show()